### Dataset Structure:

* The dataset contains 1,000,000 playlists, including playlist titles and track titles, created by users on the Spotify platform between January 2010 and October 2017

* The MPD is divided into 1,000 slice files, each containing 1,000 playlists.
    - Example file names: mpd.slice.0-999.json (first 1,000 playlists) and mpd.slice.999000-999999.json (last 1,000 playlists).

* Each file contains:
    - An info field: Metadata about the slice, such as the version, description, and license.
    - A playlists field: An array of playlists, each represented as a dictionary containing various attributes like playlist ID, name, description, number of tracks, and detailed information about each track (track name, album name, artist name, URIs, etc.).

* Each playlist contains
    - at least 5 tracks
    - no more than 250 tracks

* details structure

    - `info` Field

        The info field is a dictionary that contains general information about the particular slice:

        * **slice** - the range of slices that in in this particular file - such as 0-999
        * ***version*** -  - the current version of the MPD (which should be v1)
        * ***description*** - a description of the MPD
        * ***license*** - licensing info for the MPD
        * ***generated_on*** - a timestamp indicating when the slice was generated.

    -  `playlists` field
    
        This is an array that typically contains 1,000 playlists. Each playlist is a dictionary that contains the following fields:


        * ***pid*** - integer - playlist id - the MPD ID of this playlist. This is an integer between 0 and 999,999.
        * ***name*** - string - the name of the playlist
        * ***description*** - optional string - if present, the description given to the playlist.  Note that user-provided playlist descrptions are a relatively new feature of Spotify, so most playlists do not have descriptions.
        * ***modified_at*** - seconds - timestamp (in seconds since the epoch) when this playlist was last updated. Times are rounded to midnight GMT of the date when the playlist was last updated.
        * ***num_artists*** - the total number of unique artists for the tracks in the playlist.
        * ***num_albums*** - the number of unique albums for the tracks in the playlist
        * ***num_tracks*** - the number of tracks in the playlist
        * ***num_followers*** - the number of followers this playlist had at the time the MPD was created. (Note that the follower count does not including the playlist creator)
        * ***num_edits*** - the number of separate editing sessions. Tracks added in a two hour window are considered to be added in a single editing session.
        * ***duration_ms*** - the total duration of all the tracks in the playlist (in milliseconds)
        * ***collaborative*** -  boolean - if true, the playlist is a collaborative playlist. Multiple users may contribute tracks to a collaborative playlist.
        * ***tracks*** - an array of information about each track in the playlist. Each element in the array is a dictionary with the following fields:
        * ***track_name*** - the name of the track
        * ***track_uri*** - the Spotify URI of the track
        * ***album_name*** - the name of the track's album
        * ***album_uri*** - the Spotify URI of the album
        * ***artist_name*** - the name of the track's primary artist
        * ***artist_uri*** - the Spotify URI of track's primary artist
        * ***duration_ms*** - the duration of the track in milliseconds
        * ***pos*** - the position of the track in the playlist (zero-based)


* https://www.aicrowd.com/challenges/spotify-million-playlist-dataset-challenge

### Set Up

In [ ]:
import os
import json
import pandas as pd
import time

In [ ]:
# Path to the directory containing the JSON files
data_dir = '/Users/seoyeonpark/Desktop/DATA5011/Datasets/spotify_million_playlist_dataset/data'

### Load the Spotify Million Playliest Dataset which includes Metadata (Use 5000 playlists at the beginning)

In [ ]:
# Initialize an empty list to hold metadata and track data
playlist_metadata = []
track_metadata = []

# Define how many files to load (for the next 5 files, starting from the 6th)
start_file = 10  # Start from the 6th file (index 5 in a 0-based index)
files_to_load = 5  # Load 5 more files

# Loop through the files and extract playlist and track metadata
for i, filename in enumerate(sorted(os.listdir(data_dir))):
    if filename.endswith('.json') and start_file <= i < (start_file + files_to_load):
        file_path = os.path.join(data_dir, filename)

        # Open the JSON file and load the data
        with open(file_path, 'r') as f:
            data = json.load(f)

            # Loop through each playlist in the file
            for playlist in data['playlists']:
                # Extract playlist metadata
                playlist_info = {
                    'pid': playlist['pid'],  # Playlist ID
                    'name': playlist.get('name', ''),  # Playlist name
                    'description': playlist.get('description', ''),  # Playlist description (optional)
                    'collaborative': playlist['collaborative'],  # Collaborative status
                    'num_tracks': playlist['num_tracks'],  # Number of tracks in playlist
                    'num_followers': playlist.get('num_followers', 0),  # Number of followers
                    'num_artists': playlist['num_artists'],  # Number of unique artists
                    'num_albums': playlist['num_albums'],  # Number of unique albums
                    'duration_ms': playlist['duration_ms'],  # Total duration in milliseconds
                    'modified_at': playlist['modified_at']  # Last modified timestamp
                }
                playlist_metadata.append(playlist_info)

                # Extract track metadata from each playlist
                for track in playlist['tracks']:
                    track_info = {
                        'pid': playlist['pid'],  # Playlist ID (to link with the playlist)
                        'track_name': track['track_name'],  # Track name
                        'track_uri': track['track_uri'],  # Track URI (Spotify identifier)
                        'album_name': track['album_name'],  # Album name
                        'artist_name': track['artist_name'],  # Artist name
                        'album_uri': track['album_uri'],  # Spotify album URI
                        'artist_uri': track['artist_uri'],  # Spotify artist URI
                        'duration_ms': track['duration_ms'],  # Track duration in milliseconds
                        'pos': track['pos']  # Track position in the playlist
                    }
                    track_metadata.append(track_info)

# Convert the metadata into pandas DataFrames
df_playlist_metadata = pd.DataFrame(playlist_metadata)
df_track_metadata = pd.DataFrame(track_metadata)


In [ ]:
df_playlist_metadata


,pid,name,description,collaborative,num_tracks,num_followers,num_artists,num_albums,duration_ms,modified_at
0,107000,Sober,,false,31,2,19,29,7380816,1447372800
1,107001,Turn Up,,false,21,1,19,21,4561959,1467590400
2,107002,Alt.,,false,18,1,10,12,4058358,1485129600
3,107003,From me to you,,false,16,1,14,15,4039902,1505520000
4,107004,New Songs,,false,33,1,24,30,7392414,1509408000
...,...,...,...,...,...,...,...,...,...,...
4995,110995,//the goods//,,false,205,2,137,176,51636217,1509321600
4996,110996,Wedding Music,,true,8,1,8,8,1766170,1417651200
4997,110997,Good Vibes,,false,33,2,25,31,8387469,1475539200
4998,110998,October 16,,false,59,1,38,46,14040729,1477612800


In [ ]:
df_track_metadata

,pid,track_name,track_uri,album_name,artist_name,album_uri,artist_uri,duration_ms,pos
0,107000,I'm Different,spotify:track:6J5sxraPPZ4b0CVOGAgpXj,Based On A T.R.U. Story,2 Chainz,spotify:album:1wBFRaacNYmqfkidUZ0NtM,spotify:artist:17lzZA2AlOHwCwFALHttmp,207040,0
1,107000,Energy,spotify:track:79XrkTOfV1AqySNjVlygpW,If You're Reading This It's Too Late,Drake,spotify:album:0ptlfJfwGTy0Yvrk14JK1I,spotify:artist:3TVXtAsR1Inumwj472S9r4,181933,1
2,107000,F**kin' Problems,spotify:track:4X5f3vT8MRuXF68pfjNte5,LONG.LIVE.A$AP (Deluxe Version),A$AP Rocky,spotify:album:6rzMufuu8sLkIizM4q9c7J,spotify:artist:13ubrt8QOOCPljQ2FL1Kca,233786,2
3,107000,Under Ground Kings,spotify:track:1D9XLqQp2YYiOxrr5KLb8K,Take Care,Drake,spotify:album:6X1x82kppWZmDzlXXK3y3q,spotify:artist:3TVXtAsR1Inumwj472S9r4,212613,3
4,107000,"Trouble on My Mind (feat. Tyler, The Creator)",spotify:track:34Fulx6Umr9LoA4UKdcjVP,Fear of God II,Pusha T,spotify:album:4xKFQrfj75miFz8URY5Ft4,spotify:artist:0ONHkAv9pCAFxb0zJwDNTy,212506,4
...,...,...,...,...,...,...,...,...,...
334947,110999,Such Great Heights,spotify:track:3iDK8BAaBUatPR84gdfa9g,Give Up,The Postal Service,spotify:album:19AxGebIDoa4PrkvuZ2ACZ,spotify:artist:5yV1qdnmxyIYiSFB02wpDj,266346,76
334948,110999,Ghosts,spotify:track:00CrtqaRkCyFjY1yiSYJWo,Pacifica,The Presets,spotify:album:7lDhTfUHPUM7t1muc0NyoC,spotify:artist:1zTAQ6zkGz2L2i6lfR30EX,209053,77
334949,110999,Lofticries,spotify:track:45aZM1UAGfThfsQ6haapFu,Shrines,Purity Ring,spotify:album:7ppypgQppMf3mkRbZxYIFM,spotify:artist:1TtJ8j22Roc24e2Jx3OcU4,239613,78
334950,110999,"She Said, She Said",spotify:track:7l0JEIS70KQIZTKF6wxic5,The Big Come Up,The Black Keys,spotify:album:7DDMtj3GwKJ8HHBm18OdKT,spotify:artist:7mnBLXK823vNxN3UWB7Gfz,152253,79


In [ ]:
metadata_features = ['pid','track_name','track_uri', 'album_name', 'artist_name']

df_combined = pd.concat([df_playlist_metadata['name'], df_track_metadata.loc[:, metadata_features]], axis=1)  # axis=1 for horizontal concatenation
df_combined

,name,pid,track_name,track_uri,album_name,artist_name
0,Sober,107000,I'm Different,spotify:track:6J5sxraPPZ4b0CVOGAgpXj,Based On A T.R.U. Story,2 Chainz
1,Turn Up,107000,Energy,spotify:track:79XrkTOfV1AqySNjVlygpW,If You're Reading This It's Too Late,Drake
2,Alt.,107000,F**kin' Problems,spotify:track:4X5f3vT8MRuXF68pfjNte5,LONG.LIVE.A$AP (Deluxe Version),A$AP Rocky
3,From me to you,107000,Under Ground Kings,spotify:track:1D9XLqQp2YYiOxrr5KLb8K,Take Care,Drake
4,New Songs,107000,"Trouble on My Mind (feat. Tyler, The Creator)",spotify:track:34Fulx6Umr9LoA4UKdcjVP,Fear of God II,Pusha T
...,...,...,...,...,...,...
334947,NaN,110999,Such Great Heights,spotify:track:3iDK8BAaBUatPR84gdfa9g,Give Up,The Postal Service
334948,NaN,110999,Ghosts,spotify:track:00CrtqaRkCyFjY1yiSYJWo,Pacifica,The Presets
334949,NaN,110999,Lofticries,spotify:track:45aZM1UAGfThfsQ6haapFu,Shrines,Purity Ring
334950,NaN,110999,"She Said, She Said",spotify:track:7l0JEIS70KQIZTKF6wxic5,The Big Come Up,The Black Keys


### Extract the Audio Data

##### Audio Features:
1. **`danceability`**:
   - Describes how suitable a track is for dancing based on a combination of musical elements including tempo, rhythm stability, beat strength, and overall regularity.
   - Value range: 0.0 (least danceable) to 1.0 (most danceable).

2. **`energy`**:
   - A measure from 0.0 to 1.0 that represents a perceptual measure of intensity and activity.
   - High energy tracks feel fast, loud, and noisy, while low energy tracks feel slow and quiet.
   
3. **`key`**:
   - The key the track is in. The values are integers that map to pitches using standard Pitch Class notation (e.g., C = 0, C♯/D♭ = 1, D = 2, etc.).
   - If no key is detected, the value is `-1`.

4. **`loudness`**:
   - The overall loudness of a track in decibels (dB).
   - Loudness values are averaged across the entire track and typically range between -60 and 0 dB.

5. **`mode`**:
   - Indicates the modality (major or minor) of a track. Major is represented by 1 and minor is 0.

6. **`speechiness`**:
   - Measures the presence of spoken words in a track.
   - A value close to 1.0 means the track is almost entirely spoken words (e.g., podcasts or talk shows), while a value below 0.33 likely means the track is mostly music.

7. **`acousticness`**:
   - A measure from 0.0 to 1.0 that predicts whether a track is acoustic.
   - A value of 1.0 represents high confidence that the track is acoustic.

8. **`instrumentalness`**:
   - Predicts whether a track contains no vocals.
   - The closer the instrumentalness value is to 1.0, the more likely the track is instrumental. Values above 0.5 indicate instrumental tracks.

9. **`liveness`**:
   - Detects the presence of an audience in the recording.
   - A value above 0.8 indicates a high probability that the track was performed live.

10. **`valence`**:
    - A measure from 0.0 to 1.0 describing the musical positiveness conveyed by a track.
    - Tracks with high valence sound more positive (happy, cheerful), while tracks with low valence sound more negative (sad, angry).

11. **`tempo`**:
    - The overall estimated tempo of a track in beats per minute (BPM).
    - Tempo measures the speed or pace of a track.

12. **`duration_ms`**:
    - The duration of the track in milliseconds.

13. **`time_signature`**:
    - An estimated overall time signature of a track. The time signature is a notational convention to specify how many beats are in each bar (measure).
    - Common values are 3 (waltz) and 4 (standard pop/rock).

##### Non-Audio Features:
- **`id`**:
   - Spotify's unique ID for the track.
  
- **`uri`**:
   - Spotify's URI for the track.

- **`track_href`**:
   - A link to the Web API endpoint providing full details of the track.

- **`analysis_url`**:
   - A URL to the detailed track analysis data from Spotify’s audio analysis endpoint.

##### Metadata Columns:
- **`name`**:
   - Playlist name (from your combined DataFrame).
   
- **`pid`**:
   - Playlist ID.
   
- **`track_name`**:
   - Name of the track.

- **`track_uri`**:
   - Spotify URI for the track.

- **`album_name`**:
   - Name of the track's album.

- **`artist_name`**:
   - Name of the track’s primary artist.

#### API Setup

In [ ]:
pip install spotipy

In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

# Set up your Spotify API credentials
SPOTIPY_CLIENT_ID = ''
SPOTIPY_CLIENT_SECRET = ''




# Authenticate with Spotify
client_credentials_manager = SpotifyClientCredentials(client_id=SPOTIPY_CLIENT_ID, client_secret=SPOTIPY_CLIENT_SECRET)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)

# Test the connection by retrieving the current user's details
results = sp.search(q='The Beatles', limit=5)
for idx, track in enumerate(results['tracks']['items']):
    print(f"{idx+1}. {track['name']} - {track['artists'][0]['name']}")


1. Here Comes The Sun - Remastered 2009 - The Beatles
2. In My Life - Remastered 2009 - The Beatles
3. Yellow Submarine - The Beatles
4. Blackbird - Remastered 2009 - The Beatles
5. Come Together - Remastered 2009 - The Beatles


#### create dataframe for audio data


* USE 15 files to create the million song dataset
* Each file consist of 1000 playlists
* Total, 15000 playlists has been used out of million playlists.


In [ ]:
# Reduce the original dataframe for testing purposes
#df_combined_reduced  = df_combined
df_combined_reduced  = df_combined.iloc[200000:334952]

# Function to fetch audio features in batches and retain None for missing values
def fetch_audio_features(df, batch_size=100):
    track_uris = df['track_uri'].tolist()  # Get the list of track URIs
    audio_features_list = []  # To store audio features

    for i in range(0, len(track_uris), batch_size):
        batch = track_uris[i:i+batch_size]  # Get a batch of URIs
        try:
            features = sp.audio_features(batch)  # Fetch audio features for this batch
            # For each URI in the batch, add features or None if no data is available
            for j, track_uri in enumerate(batch):
                if features and features[j] is not None:  # Only append valid features
                    # Include the 'track_uri' in the features dictionary to use as a key for merging
                    features[j]['track_uri'] = track_uri
                    audio_features_list.append(features[j])
                else:
                    # If no features were returned, append None values for that track
                    audio_features_list.append({'track_uri': track_uri, 'danceability': None, 'energy': None, 'key': None, 'loudness': None, 'mode': None, 'speechiness': None, 'acousticness': None, 'instrumentalness': None, 'liveness': None, 'valence': None, 'tempo': None, 'type': None, 'id': None, 'uri': None, 'track_href': None, 'analysis_url': None, 'duration_ms': None, 'time_signature': None})
            print(f"Processed batch {i//batch_size + 1}")
        except Exception as e:
            print(f"Error fetching audio features: {e}")
            time.sleep(1)  # In case of an error, wait for a second and continue

    return audio_features_list

# Fetch audio features in batches of 100
audio_features = fetch_audio_features(df_combined_reduced)

# Convert the list of audio features to a DataFrame
df_audio_features = pd.DataFrame(audio_features)

# Concatenate the original DataFrame with the audio features horizontally (axis=1)
df_combined_with_features = pd.concat([df_combined_reduced.reset_index(drop=True), df_audio_features.reset_index(drop=True)], axis=1)

# Display the final DataFrame with audio features
print(df_combined_with_features.head())


In [ ]:
df_15 = df_combined_with_features
df_15

,name,pid,track_name,track_uri,album_name,artist_name,danceability,energy,key,loudness,...,valence,tempo,type,id,uri,track_href,analysis_url,duration_ms,time_signature,track_uri
0,NaN,109949,I Am A Man Of Constant Sorrow - From “O Brothe...,spotify:track:5AhDb4oM6f4YmHPXW123Fg,"O Brother, Where Art Thou?",The Soggy Bottom Boys,0.568,0.308,5.0,-13.063,...,0.721,85.917,audio_features,5AhDb4oM6f4YmHPXW123Fg,spotify:track:5AhDb4oM6f4YmHPXW123Fg,https://api.spotify.com/v1/tracks/5AhDb4oM6f4Y...,https://api.spotify.com/v1/audio-analysis/5AhD...,190000.0,4.0,spotify:track:5AhDb4oM6f4YmHPXW123Fg
1,NaN,109949,She's Like The Wind - Remastered 2003,spotify:track:3aNYqoYOOtZWAnEvyyymtF,Ultimate Dirty Dancing,Patrick Swayze Featuring Wendy Fraser,0.570,0.526,4.0,-5.772,...,0.142,125.218,audio_features,3aNYqoYOOtZWAnEvyyymtF,spotify:track:3aNYqoYOOtZWAnEvyyymtF,https://api.spotify.com/v1/tracks/3aNYqoYOOtZW...,https://api.spotify.com/v1/audio-analysis/3aNY...,231853.0,4.0,spotify:track:3aNYqoYOOtZWAnEvyyymtF
2,NaN,109949,Every Breath You Take - Remastered 2003,spotify:track:5C0LFQARavkPpn7JgA4sLk,Synchronicity,The Police,0.820,0.452,1.0,-9.796,...,0.740,117.401,audio_features,5C0LFQARavkPpn7JgA4sLk,spotify:track:5C0LFQARavkPpn7JgA4sLk,https://api.spotify.com/v1/tracks/5C0LFQARavkP...,https://api.spotify.com/v1/audio-analysis/5C0L...,253250.0,4.0,spotify:track:5C0LFQARavkPpn7JgA4sLk
3,NaN,109949,Don't Stop Believin',spotify:track:4bHsxqR3GMrXTxEPLuK5ue,Escape,Journey,0.500,0.748,4.0,-9.072,...,0.514,118.852,audio_features,4bHsxqR3GMrXTxEPLuK5ue,spotify:track:4bHsxqR3GMrXTxEPLuK5ue,https://api.spotify.com/v1/tracks/4bHsxqR3GMrX...,https://api.spotify.com/v1/audio-analysis/4bHs...,250987.0,4.0,spotify:track:4bHsxqR3GMrXTxEPLuK5ue
4,NaN,109949,Sangria,spotify:track:0p1HtkrNYxv0iDfEKwXSTp,BRINGING BACK THE SUNSHINE,Blake Shelton,0.646,0.724,9.0,-6.960,...,0.540,115.984,audio_features,0p1HtkrNYxv0iDfEKwXSTp,spotify:track:0p1HtkrNYxv0iDfEKwXSTp,https://api.spotify.com/v1/tracks/0p1HtkrNYxv0...,https://api.spotify.com/v1/audio-analysis/0p1H...,233480.0,4.0,spotify:track:0p1HtkrNYxv0iDfEKwXSTp
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134947,NaN,110999,Such Great Heights,spotify:track:3iDK8BAaBUatPR84gdfa9g,Give Up,The Postal Service,0.653,0.818,5.0,-8.125,...,0.198,174.984,audio_features,3iDK8BAaBUatPR84gdfa9g,spotify:track:3iDK8BAaBUatPR84gdfa9g,https://api.spotify.com/v1/tracks/3iDK8BAaBUat...,https://api.spotify.com/v1/audio-analysis/3iDK...,266347.0,4.0,spotify:track:3iDK8BAaBUatPR84gdfa9g
134948,NaN,110999,Ghosts,spotify:track:00CrtqaRkCyFjY1yiSYJWo,Pacifica,The Presets,0.610,0.683,1.0,-9.137,...,0.582,140.083,audio_features,00CrtqaRkCyFjY1yiSYJWo,spotify:track:00CrtqaRkCyFjY1yiSYJWo,https://api.spotify.com/v1/tracks/00CrtqaRkCyF...,https://api.spotify.com/v1/audio-analysis/00Cr...,209053.0,4.0,spotify:track:00CrtqaRkCyFjY1yiSYJWo
134949,NaN,110999,Lofticries,spotify:track:45aZM1UAGfThfsQ6haapFu,Shrines,Purity Ring,0.671,0.730,8.0,-6.773,...,0.299,112.011,audio_features,45aZM1UAGfThfsQ6haapFu,spotify:track:45aZM1UAGfThfsQ6haapFu,https://api.spotify.com/v1/tracks/45aZM1UAGfTh...,https://api.spotify.com/v1/audio-analysis/45aZ...,239613.0,4.0,spotify:track:45aZM1UAGfThfsQ6haapFu
134950,NaN,110999,"She Said, She Said",spotify:track:7l0JEIS70KQIZTKF6wxic5,The Big Come Up,The Black Keys,0.500,0.775,2.0,-3.394,...,0.930,123.094,audio_features,7l0JEIS70KQIZTKF6wxic5,spotify:track:7l0JEIS70KQIZTKF6wxic5,https://api.spotify.com/v1/tracks/7l0JEIS70KQI...,https://api.spotify.com/v1/audio-analysis/7l0J...,152253.0,4.0,spotify:track:7l0JEIS70KQIZTKF6wxic5


In [ ]:
df_0_870568 = pd.read_csv('df_0_870568.csv')
df_0_870568 = df_0_870568.rename(columns={'track_uri.1' : 'track_uri'})
df_0_870568

/var/folders/ps/5d5cgvy91xv6hkty0gc2y0g00000gn/T/ipykernel_65463/3540801520.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_0_870568 = pd.read_csv('df_0_870568.csv')


,name,pid,track_name,track_uri,album_name,artist_name,danceability,energy,key,loudness,...,valence,tempo,type,id,uri,track_href,analysis_url,duration_ms,time_signature,track_uri
0,Throwbacks,0,Lose Control (feat. Ciara & Fat Man Scoop),spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,The Cookbook,Missy Elliott,0.904,0.813,4.0,-7.105,...,0.810,125.461,audio_features,0UaMYEvWZi0ZqiDOoHU3YI,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,https://api.spotify.com/v1/tracks/0UaMYEvWZi0Z...,https://api.spotify.com/v1/audio-analysis/0UaM...,226864.0,4.0,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI
1,Awesome Playlist,0,Toxic,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,In The Zone,Britney Spears,0.774,0.838,5.0,-3.914,...,0.924,143.040,audio_features,6I9VzXrHxO9rA9A5euc8Ak,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,https://api.spotify.com/v1/tracks/6I9VzXrHxO9r...,https://api.spotify.com/v1/audio-analysis/6I9V...,198800.0,4.0,spotify:track:6I9VzXrHxO9rA9A5euc8Ak
2,korean,0,Crazy In Love,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,Dangerously In Love (Alben für die Ewigkeit),Beyoncé,0.664,0.759,2.0,-6.583,...,0.701,99.252,audio_features,0WqIKmW4BTrj3eJFmnCKMv,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,https://api.spotify.com/v1/tracks/0WqIKmW4BTrj...,https://api.spotify.com/v1/audio-analysis/0WqI...,235933.0,4.0,spotify:track:0WqIKmW4BTrj3eJFmnCKMv
3,mat,0,Rock Your Body,spotify:track:1AWQoqb9bSvzTjaLralEkT,Justified,Justin Timberlake,0.892,0.714,4.0,-6.055,...,0.817,100.972,audio_features,1AWQoqb9bSvzTjaLralEkT,spotify:track:1AWQoqb9bSvzTjaLralEkT,https://api.spotify.com/v1/tracks/1AWQoqb9bSvz...,https://api.spotify.com/v1/audio-analysis/1AWQ...,267267.0,4.0,spotify:track:1AWQoqb9bSvzTjaLralEkT
4,90s,0,It Wasn't Me,spotify:track:1lzr43nnXAijIGYnCT8M8H,Hot Shot,Shaggy,0.853,0.606,0.0,-4.596,...,0.654,94.759,audio_features,1lzr43nnXAijIGYnCT8M8H,spotify:track:1lzr43nnXAijIGYnCT8M8H,https://api.spotify.com/v1/tracks/1lzr43nnXAij...,https://api.spotify.com/v1/audio-analysis/1lzr...,227600.0,4.0,spotify:track:1lzr43nnXAijIGYnCT8M8H
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
870563,NaN,109949,Dancing In The Street,spotify:track:3UoUMTLrAmjlHSmuC8aEKq,Dance Party,Martha Reeves & The Vandellas,0.557,0.578,2.0,-9.423,...,0.292,126.019,audio_features,3UoUMTLrAmjlHSmuC8aEKq,spotify:track:3UoUMTLrAmjlHSmuC8aEKq,https://api.spotify.com/v1/tracks/3UoUMTLrAmjl...,https://api.spotify.com/v1/audio-analysis/3UoU...,165227.0,4.0,spotify:track:3UoUMTLrAmjlHSmuC8aEKq
870564,NaN,109949,Drinking class,spotify:track:5Kb5utdaR6qXYZ9WpgoKWm,I Don't Dance,Lee Brice,0.663,0.530,1.0,-9.763,...,0.269,109.003,audio_features,5Kb5utdaR6qXYZ9WpgoKWm,spotify:track:5Kb5utdaR6qXYZ9WpgoKWm,https://api.spotify.com/v1/tracks/5Kb5utdaR6qX...,https://api.spotify.com/v1/audio-analysis/5Kb5...,207347.0,4.0,spotify:track:5Kb5utdaR6qXYZ9WpgoKWm
870565,NaN,109949,Say You Do,spotify:track:2sE61ZmvYH8wiOx5jygkHH,RISER,Dierks Bentley,0.481,0.733,3.0,-8.171,...,0.453,147.859,audio_features,2sE61ZmvYH8wiOx5jygkHH,spotify:track:2sE61ZmvYH8wiOx5jygkHH,https://api.spotify.com/v1/tracks/2sE61ZmvYH8w...,https://api.spotify.com/v1/audio-analysis/2sE6...,219360.0,4.0,spotify:track:2sE61ZmvYH8wiOx5jygkHH
870566,NaN,109949,Bartender,spotify:track:6olKv2HP3XgBpvVxAswowe,747,Lady Antebellum,0.631,0.937,11.0,-3.858,...,0.654,101.011,audio_features,6olKv2HP3XgBpvVxAswowe,spotify:track:6olKv2HP3XgBpvVxAswowe,https://api.spotify.com/v1/tracks/6olKv2HP3XgB...,https://api.spotify.com/v1/audio-analysis/6olK...,198267.0,4.0,spotify:track:6olKv2HP3XgBpvVxAswowe


In [ ]:
df_million = pd.concat([df_0_870568, df_15], axis=0)
df_million.to_csv('df_million.csv', index=False)
df_million

,name,pid,track_name,track_uri,album_name,artist_name,danceability,energy,key,loudness,...,valence,tempo,type,id,uri,track_href,analysis_url,duration_ms,time_signature,track_uri
0,Throwbacks,0,Lose Control (feat. Ciara & Fat Man Scoop),spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,The Cookbook,Missy Elliott,0.904,0.813,4.0,-7.105,...,0.810,125.461,audio_features,0UaMYEvWZi0ZqiDOoHU3YI,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,https://api.spotify.com/v1/tracks/0UaMYEvWZi0Z...,https://api.spotify.com/v1/audio-analysis/0UaM...,226864.0,4.0,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI
1,Awesome Playlist,0,Toxic,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,In The Zone,Britney Spears,0.774,0.838,5.0,-3.914,...,0.924,143.040,audio_features,6I9VzXrHxO9rA9A5euc8Ak,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,https://api.spotify.com/v1/tracks/6I9VzXrHxO9r...,https://api.spotify.com/v1/audio-analysis/6I9V...,198800.0,4.0,spotify:track:6I9VzXrHxO9rA9A5euc8Ak
2,korean,0,Crazy In Love,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,Dangerously In Love (Alben für die Ewigkeit),Beyoncé,0.664,0.759,2.0,-6.583,...,0.701,99.252,audio_features,0WqIKmW4BTrj3eJFmnCKMv,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,https://api.spotify.com/v1/tracks/0WqIKmW4BTrj...,https://api.spotify.com/v1/audio-analysis/0WqI...,235933.0,4.0,spotify:track:0WqIKmW4BTrj3eJFmnCKMv
3,mat,0,Rock Your Body,spotify:track:1AWQoqb9bSvzTjaLralEkT,Justified,Justin Timberlake,0.892,0.714,4.0,-6.055,...,0.817,100.972,audio_features,1AWQoqb9bSvzTjaLralEkT,spotify:track:1AWQoqb9bSvzTjaLralEkT,https://api.spotify.com/v1/tracks/1AWQoqb9bSvz...,https://api.spotify.com/v1/audio-analysis/1AWQ...,267267.0,4.0,spotify:track:1AWQoqb9bSvzTjaLralEkT
4,90s,0,It Wasn't Me,spotify:track:1lzr43nnXAijIGYnCT8M8H,Hot Shot,Shaggy,0.853,0.606,0.0,-4.596,...,0.654,94.759,audio_features,1lzr43nnXAijIGYnCT8M8H,spotify:track:1lzr43nnXAijIGYnCT8M8H,https://api.spotify.com/v1/tracks/1lzr43nnXAij...,https://api.spotify.com/v1/audio-analysis/1lzr...,227600.0,4.0,spotify:track:1lzr43nnXAijIGYnCT8M8H
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134947,NaN,110999,Such Great Heights,spotify:track:3iDK8BAaBUatPR84gdfa9g,Give Up,The Postal Service,0.653,0.818,5.0,-8.125,...,0.198,174.984,audio_features,3iDK8BAaBUatPR84gdfa9g,spotify:track:3iDK8BAaBUatPR84gdfa9g,https://api.spotify.com/v1/tracks/3iDK8BAaBUat...,https://api.spotify.com/v1/audio-analysis/3iDK...,266347.0,4.0,spotify:track:3iDK8BAaBUatPR84gdfa9g
134948,NaN,110999,Ghosts,spotify:track:00CrtqaRkCyFjY1yiSYJWo,Pacifica,The Presets,0.610,0.683,1.0,-9.137,...,0.582,140.083,audio_features,00CrtqaRkCyFjY1yiSYJWo,spotify:track:00CrtqaRkCyFjY1yiSYJWo,https://api.spotify.com/v1/tracks/00CrtqaRkCyF...,https://api.spotify.com/v1/audio-analysis/00Cr...,209053.0,4.0,spotify:track:00CrtqaRkCyFjY1yiSYJWo
134949,NaN,110999,Lofticries,spotify:track:45aZM1UAGfThfsQ6haapFu,Shrines,Purity Ring,0.671,0.730,8.0,-6.773,...,0.299,112.011,audio_features,45aZM1UAGfThfsQ6haapFu,spotify:track:45aZM1UAGfThfsQ6haapFu,https://api.spotify.com/v1/tracks/45aZM1UAGfTh...,https://api.spotify.com/v1/audio-analysis/45aZ...,239613.0,4.0,spotify:track:45aZM1UAGfThfsQ6haapFu
134950,NaN,110999,"She Said, She Said",spotify:track:7l0JEIS70KQIZTKF6wxic5,The Big Come Up,The Black Keys,0.500,0.775,2.0,-3.394,...,0.930,123.094,audio_features,7l0JEIS70KQIZTKF6wxic5,spotify:track:7l0JEIS70KQIZTKF6wxic5,https://api.spotify.com/v1/tracks/7l0JEIS70KQI...,https://api.spotify.com/v1/audio-analysis/7l0J...,152253.0,4.0,spotify:track:7l0JEIS70KQIZTKF6wxic5


In [ ]:
list(df_million.columns)

['name',
 'pid',
 'track_name',
 'track_uri',
 'album_name',
 'artist_name',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'type',
 'id',
 'uri',
 'track_href',
 'analysis_url',
 'duration_ms',
 'time_signature',
 'track_uri']

In [ ]:
df = pd.read_csv('df_million.csv')
df

/var/folders/ps/5d5cgvy91xv6hkty0gc2y0g00000gn/T/ipykernel_65463/3522353669.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('df_million.csv')


,name,pid,track_name,track_uri,album_name,artist_name,danceability,energy,key,loudness,...,valence,tempo,type,id,uri,track_href,analysis_url,duration_ms,time_signature,track_uri.1
0,Throwbacks,0,Lose Control (feat. Ciara & Fat Man Scoop),spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,The Cookbook,Missy Elliott,0.904,0.813,4.0,-7.105,...,0.810,125.461,audio_features,0UaMYEvWZi0ZqiDOoHU3YI,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,https://api.spotify.com/v1/tracks/0UaMYEvWZi0Z...,https://api.spotify.com/v1/audio-analysis/0UaM...,226864.0,4.0,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI
1,Awesome Playlist,0,Toxic,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,In The Zone,Britney Spears,0.774,0.838,5.0,-3.914,...,0.924,143.040,audio_features,6I9VzXrHxO9rA9A5euc8Ak,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,https://api.spotify.com/v1/tracks/6I9VzXrHxO9r...,https://api.spotify.com/v1/audio-analysis/6I9V...,198800.0,4.0,spotify:track:6I9VzXrHxO9rA9A5euc8Ak
2,korean,0,Crazy In Love,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,Dangerously In Love (Alben für die Ewigkeit),Beyoncé,0.664,0.759,2.0,-6.583,...,0.701,99.252,audio_features,0WqIKmW4BTrj3eJFmnCKMv,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,https://api.spotify.com/v1/tracks/0WqIKmW4BTrj...,https://api.spotify.com/v1/audio-analysis/0WqI...,235933.0,4.0,spotify:track:0WqIKmW4BTrj3eJFmnCKMv
3,mat,0,Rock Your Body,spotify:track:1AWQoqb9bSvzTjaLralEkT,Justified,Justin Timberlake,0.892,0.714,4.0,-6.055,...,0.817,100.972,audio_features,1AWQoqb9bSvzTjaLralEkT,spotify:track:1AWQoqb9bSvzTjaLralEkT,https://api.spotify.com/v1/tracks/1AWQoqb9bSvz...,https://api.spotify.com/v1/audio-analysis/1AWQ...,267267.0,4.0,spotify:track:1AWQoqb9bSvzTjaLralEkT
4,90s,0,It Wasn't Me,spotify:track:1lzr43nnXAijIGYnCT8M8H,Hot Shot,Shaggy,0.853,0.606,0.0,-4.596,...,0.654,94.759,audio_features,1lzr43nnXAijIGYnCT8M8H,spotify:track:1lzr43nnXAijIGYnCT8M8H,https://api.spotify.com/v1/tracks/1lzr43nnXAij...,https://api.spotify.com/v1/audio-analysis/1lzr...,227600.0,4.0,spotify:track:1lzr43nnXAijIGYnCT8M8H
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1005515,NaN,110999,Such Great Heights,spotify:track:3iDK8BAaBUatPR84gdfa9g,Give Up,The Postal Service,0.653,0.818,5.0,-8.125,...,0.198,174.984,audio_features,3iDK8BAaBUatPR84gdfa9g,spotify:track:3iDK8BAaBUatPR84gdfa9g,https://api.spotify.com/v1/tracks/3iDK8BAaBUat...,https://api.spotify.com/v1/audio-analysis/3iDK...,266347.0,4.0,spotify:track:3iDK8BAaBUatPR84gdfa9g
1005516,NaN,110999,Ghosts,spotify:track:00CrtqaRkCyFjY1yiSYJWo,Pacifica,The Presets,0.610,0.683,1.0,-9.137,...,0.582,140.083,audio_features,00CrtqaRkCyFjY1yiSYJWo,spotify:track:00CrtqaRkCyFjY1yiSYJWo,https://api.spotify.com/v1/tracks/00CrtqaRkCyF...,https://api.spotify.com/v1/audio-analysis/00Cr...,209053.0,4.0,spotify:track:00CrtqaRkCyFjY1yiSYJWo
1005517,NaN,110999,Lofticries,spotify:track:45aZM1UAGfThfsQ6haapFu,Shrines,Purity Ring,0.671,0.730,8.0,-6.773,...,0.299,112.011,audio_features,45aZM1UAGfThfsQ6haapFu,spotify:track:45aZM1UAGfThfsQ6haapFu,https://api.spotify.com/v1/tracks/45aZM1UAGfTh...,https://api.spotify.com/v1/audio-analysis/45aZ...,239613.0,4.0,spotify:track:45aZM1UAGfThfsQ6haapFu
1005518,NaN,110999,"She Said, She Said",spotify:track:7l0JEIS70KQIZTKF6wxic5,The Big Come Up,The Black Keys,0.500,0.775,2.0,-3.394,...,0.930,123.094,audio_features,7l0JEIS70KQIZTKF6wxic5,spotify:track:7l0JEIS70KQIZTKF6wxic5,https://api.spotify.com/v1/tracks/7l0JEIS70KQI...,https://api.spotify.com/v1/audio-analysis/7l0J...,152253.0,4.0,spotify:track:7l0JEIS70KQIZTKF6wxic5


In [ ]:
df = df.drop(columns=['name'])
df

,pid,track_name,track_uri,album_name,artist_name,danceability,energy,key,loudness,mode,...,valence,tempo,type,id,uri,track_href,analysis_url,duration_ms,time_signature,track_uri.1
0,0,Lose Control (feat. Ciara & Fat Man Scoop),spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,The Cookbook,Missy Elliott,0.904,0.813,4.0,-7.105,0.0,...,0.810,125.461,audio_features,0UaMYEvWZi0ZqiDOoHU3YI,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,https://api.spotify.com/v1/tracks/0UaMYEvWZi0Z...,https://api.spotify.com/v1/audio-analysis/0UaM...,226864.0,4.0,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI
1,0,Toxic,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,In The Zone,Britney Spears,0.774,0.838,5.0,-3.914,0.0,...,0.924,143.040,audio_features,6I9VzXrHxO9rA9A5euc8Ak,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,https://api.spotify.com/v1/tracks/6I9VzXrHxO9r...,https://api.spotify.com/v1/audio-analysis/6I9V...,198800.0,4.0,spotify:track:6I9VzXrHxO9rA9A5euc8Ak
2,0,Crazy In Love,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,Dangerously In Love (Alben für die Ewigkeit),Beyoncé,0.664,0.759,2.0,-6.583,0.0,...,0.701,99.252,audio_features,0WqIKmW4BTrj3eJFmnCKMv,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,https://api.spotify.com/v1/tracks/0WqIKmW4BTrj...,https://api.spotify.com/v1/audio-analysis/0WqI...,235933.0,4.0,spotify:track:0WqIKmW4BTrj3eJFmnCKMv
3,0,Rock Your Body,spotify:track:1AWQoqb9bSvzTjaLralEkT,Justified,Justin Timberlake,0.892,0.714,4.0,-6.055,0.0,...,0.817,100.972,audio_features,1AWQoqb9bSvzTjaLralEkT,spotify:track:1AWQoqb9bSvzTjaLralEkT,https://api.spotify.com/v1/tracks/1AWQoqb9bSvz...,https://api.spotify.com/v1/audio-analysis/1AWQ...,267267.0,4.0,spotify:track:1AWQoqb9bSvzTjaLralEkT
4,0,It Wasn't Me,spotify:track:1lzr43nnXAijIGYnCT8M8H,Hot Shot,Shaggy,0.853,0.606,0.0,-4.596,1.0,...,0.654,94.759,audio_features,1lzr43nnXAijIGYnCT8M8H,spotify:track:1lzr43nnXAijIGYnCT8M8H,https://api.spotify.com/v1/tracks/1lzr43nnXAij...,https://api.spotify.com/v1/audio-analysis/1lzr...,227600.0,4.0,spotify:track:1lzr43nnXAijIGYnCT8M8H
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1005515,110999,Such Great Heights,spotify:track:3iDK8BAaBUatPR84gdfa9g,Give Up,The Postal Service,0.653,0.818,5.0,-8.125,1.0,...,0.198,174.984,audio_features,3iDK8BAaBUatPR84gdfa9g,spotify:track:3iDK8BAaBUatPR84gdfa9g,https://api.spotify.com/v1/tracks/3iDK8BAaBUat...,https://api.spotify.com/v1/audio-analysis/3iDK...,266347.0,4.0,spotify:track:3iDK8BAaBUatPR84gdfa9g
1005516,110999,Ghosts,spotify:track:00CrtqaRkCyFjY1yiSYJWo,Pacifica,The Presets,0.610,0.683,1.0,-9.137,1.0,...,0.582,140.083,audio_features,00CrtqaRkCyFjY1yiSYJWo,spotify:track:00CrtqaRkCyFjY1yiSYJWo,https://api.spotify.com/v1/tracks/00CrtqaRkCyF...,https://api.spotify.com/v1/audio-analysis/00Cr...,209053.0,4.0,spotify:track:00CrtqaRkCyFjY1yiSYJWo
1005517,110999,Lofticries,spotify:track:45aZM1UAGfThfsQ6haapFu,Shrines,Purity Ring,0.671,0.730,8.0,-6.773,1.0,...,0.299,112.011,audio_features,45aZM1UAGfThfsQ6haapFu,spotify:track:45aZM1UAGfThfsQ6haapFu,https://api.spotify.com/v1/tracks/45aZM1UAGfTh...,https://api.spotify.com/v1/audio-analysis/45aZ...,239613.0,4.0,spotify:track:45aZM1UAGfThfsQ6haapFu
1005518,110999,"She Said, She Said",spotify:track:7l0JEIS70KQIZTKF6wxic5,The Big Come Up,The Black Keys,0.500,0.775,2.0,-3.394,1.0,...,0.930,123.094,audio_features,7l0JEIS70KQIZTKF6wxic5,spotify:track:7l0JEIS70KQIZTKF6wxic5,https://api.spotify.com/v1/tracks/7l0JEIS70KQI...,https://api.spotify.com/v1/audio-analysis/7l0J...,152253.0,4.0,spotify:track:7l0JEIS70KQIZTKF6wxic5


In [ ]:
list(df.columns)

['pid',
 'track_name',
 'track_uri',
 'album_name',
 'artist_name',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'type',
 'id',
 'uri',
 'track_href',
 'analysis_url',
 'duration_ms',
 'time_signature',
 'track_uri.1']

In [ ]:
df.iloc[50:100,:]

,pid,track_name,track_uri,album_name,artist_name,danceability,energy,key,loudness,mode,...,valence,tempo,type,id,uri,track_href,analysis_url,duration_ms,time_signature,track_uri.1
50,0,Check Yes Juliet,spotify:track:1b7vg5T9YKR3NNqXfBYRF7,We The Kings,We The Kings,0.3500,0.909,2.0,-4.204,1.0,...,0.314,166.866,audio_features,1b7vg5T9YKR3NNqXfBYRF7,spotify:track:1b7vg5T9YKR3NNqXfBYRF7,https://api.spotify.com/v1/tracks/1b7vg5T9YKR3...,https://api.spotify.com/v1/audio-analysis/1b7v...,220133.0,4.0,spotify:track:1b7vg5T9YKR3NNqXfBYRF7
51,0,The Great Escape,spotify:track:6GIrIt2M39wEGwjCQjGChX,Boys Like Girls,Boys Like Girls,0.4230,0.940,1.0,-4.012,0.0,...,0.505,149.934,audio_features,6GIrIt2M39wEGwjCQjGChX,spotify:track:6GIrIt2M39wEGwjCQjGChX,https://api.spotify.com/v1/tracks/6GIrIt2M39wE...,https://api.spotify.com/v1/audio-analysis/6GIr...,206520.0,4.0,spotify:track:6GIrIt2M39wEGwjCQjGChX
52,1,Eye of the Tiger,spotify:track:2HHtWyy5CgaQbC7XSoOb0e,Eye Of The Tiger,Survivor,0.8150,0.438,0.0,-14.522,0.0,...,0.552,108.965,audio_features,2HHtWyy5CgaQbC7XSoOb0e,spotify:track:2HHtWyy5CgaQbC7XSoOb0e,https://api.spotify.com/v1/tracks/2HHtWyy5CgaQ...,https://api.spotify.com/v1/audio-analysis/2HHt...,243773.0,4.0,spotify:track:2HHtWyy5CgaQbC7XSoOb0e
53,1,Libera Me From Hell (Tengen Toppa Gurren Lagann),spotify:track:1MYYt7h6amcrauCOoso3Gx,Versus Hollywood,Daniel Tidwell,0.3430,0.975,1.0,-2.502,0.0,...,0.133,119.883,audio_features,1MYYt7h6amcrauCOoso3Gx,spotify:track:1MYYt7h6amcrauCOoso3Gx,https://api.spotify.com/v1/tracks/1MYYt7h6amcr...,https://api.spotify.com/v1/audio-analysis/1MYY...,70294.0,4.0,spotify:track:1MYYt7h6amcrauCOoso3Gx
54,1,Pokémon Theme,spotify:track:3x2mJ2bjCIU70NrH49CtYR,Versus Hollywood,Daniel Tidwell,0.4140,0.959,7.0,-4.299,0.0,...,0.589,145.911,audio_features,3x2mJ2bjCIU70NrH49CtYR,spotify:track:3x2mJ2bjCIU70NrH49CtYR,https://api.spotify.com/v1/tracks/3x2mJ2bjCIU7...,https://api.spotify.com/v1/audio-analysis/3x2m...,65306.0,4.0,spotify:track:3x2mJ2bjCIU70NrH49CtYR
55,1,Concerning Hobbits (The Lord of the Rings),spotify:track:1Pm3fq1SC6lUlNVBGZi3Em,Versus Hollywood,Daniel Tidwell,0.5220,0.205,2.0,-7.986,1.0,...,0.353,103.868,audio_features,1Pm3fq1SC6lUlNVBGZi3Em,spotify:track:1Pm3fq1SC6lUlNVBGZi3Em,https://api.spotify.com/v1/tracks/1Pm3fq1SC6lU...,https://api.spotify.com/v1/audio-analysis/1Pm3...,108532.0,4.0,spotify:track:1Pm3fq1SC6lUlNVBGZi3Em
56,1,The Blood of Cuchulainn (The Boondock Saints),spotify:track:1NXTEkIeRL59NK61QuhYUl,Versus Hollywood,Daniel Tidwell,0.0849,0.956,2.0,-4.196,1.0,...,0.309,185.397,audio_features,1NXTEkIeRL59NK61QuhYUl,spotify:track:1NXTEkIeRL59NK61QuhYUl,https://api.spotify.com/v1/tracks/1NXTEkIeRL59...,https://api.spotify.com/v1/audio-analysis/1NXT...,214269.0,3.0,spotify:track:1NXTEkIeRL59NK61QuhYUl
57,1,He's a Pirate (Pirates of the Caribbean),spotify:track:3RGlJJFkWEavxeRQr9ivAd,Versus Hollywood,Daniel Tidwell,0.2830,0.925,3.0,-4.700,1.0,...,0.368,99.450,audio_features,3RGlJJFkWEavxeRQr9ivAd,spotify:track:3RGlJJFkWEavxeRQr9ivAd,https://api.spotify.com/v1/tracks/3RGlJJFkWEav...,https://api.spotify.com/v1/audio-analysis/3RGl...,110219.0,4.0,spotify:track:3RGlJJFkWEavxeRQr9ivAd
58,1,Very Bloody Tears (Castlevania II: Simon's Quest),spotify:track:0e9hR1vTrzlUvFH5PgA9rY,Versus Video Games,Daniel Tidwell,0.4920,0.970,4.0,-5.677,0.0,...,0.556,120.005,audio_features,0e9hR1vTrzlUvFH5PgA9rY,spotify:track:0e9hR1vTrzlUvFH5PgA9rY,https://api.spotify.com/v1/tracks/0e9hR1vTrzlU...,https://api.spotify.com/v1/audio-analysis/0e9h...,207520.0,4.0,spotify:track:0e9hR1vTrzlUvFH5PgA9rY
59,1,U.N. Owen Was Her? (Remix),spotify:track:7dkbEHIMLoeuG4zXGmzhEH,"Video Game Remixes, Vol. 1",Kaleptik,0.4930,0.969,2.0,-3.282,0.0,...,0.340,170.581,audio_features,7dkbEHIMLoeuG4zXGmzhEH,spotify:track:7dkbEHIMLoeuG4zXGmzhEH,https://api.spotify.com/v1/tracks/7dkbEHIMLoeu...,https://api.spotify.com/v1/audio-analysis/7dkb...,226000.0,4.0,spotify:track:7dkbEHIMLoeuG4zXGmzhEH


In [ ]:
df1 = pd.read_csv('df_0_100000.csv')
df1

/var/folders/ps/5d5cgvy91xv6hkty0gc2y0g00000gn/T/ipykernel_65463/3770529832.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv('df_0_100000.csv')


,name,pid,track_name,track_uri,album_name,artist_name,danceability,energy,key,loudness,...,valence,tempo,type,id,uri,track_href,analysis_url,duration_ms,time_signature,track_uri.1
0,Throwbacks,0,Lose Control (feat. Ciara & Fat Man Scoop),spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,The Cookbook,Missy Elliott,0.904,0.813,4.0,-7.105,...,0.810,125.461,audio_features,0UaMYEvWZi0ZqiDOoHU3YI,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,https://api.spotify.com/v1/tracks/0UaMYEvWZi0Z...,https://api.spotify.com/v1/audio-analysis/0UaM...,226864.0,4.0,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI
1,Awesome Playlist,0,Toxic,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,In The Zone,Britney Spears,0.774,0.838,5.0,-3.914,...,0.924,143.040,audio_features,6I9VzXrHxO9rA9A5euc8Ak,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,https://api.spotify.com/v1/tracks/6I9VzXrHxO9r...,https://api.spotify.com/v1/audio-analysis/6I9V...,198800.0,4.0,spotify:track:6I9VzXrHxO9rA9A5euc8Ak
2,korean,0,Crazy In Love,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,Dangerously In Love (Alben für die Ewigkeit),Beyoncé,0.664,0.759,2.0,-6.583,...,0.701,99.252,audio_features,0WqIKmW4BTrj3eJFmnCKMv,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,https://api.spotify.com/v1/tracks/0WqIKmW4BTrj...,https://api.spotify.com/v1/audio-analysis/0WqI...,235933.0,4.0,spotify:track:0WqIKmW4BTrj3eJFmnCKMv
3,mat,0,Rock Your Body,spotify:track:1AWQoqb9bSvzTjaLralEkT,Justified,Justin Timberlake,0.892,0.714,4.0,-6.055,...,0.817,100.972,audio_features,1AWQoqb9bSvzTjaLralEkT,spotify:track:1AWQoqb9bSvzTjaLralEkT,https://api.spotify.com/v1/tracks/1AWQoqb9bSvz...,https://api.spotify.com/v1/audio-analysis/1AWQ...,267267.0,4.0,spotify:track:1AWQoqb9bSvzTjaLralEkT
4,90s,0,It Wasn't Me,spotify:track:1lzr43nnXAijIGYnCT8M8H,Hot Shot,Shaggy,0.853,0.606,0.0,-4.596,...,0.654,94.759,audio_features,1lzr43nnXAijIGYnCT8M8H,spotify:track:1lzr43nnXAijIGYnCT8M8H,https://api.spotify.com/v1/tracks/1lzr43nnXAij...,https://api.spotify.com/v1/audio-analysis/1lzr...,227600.0,4.0,spotify:track:1lzr43nnXAijIGYnCT8M8H
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,NaN,1489,Praying,spotify:track:0jdny0dhgjUwoIp5GkqEaA,Rainbow,Kesha,0.543,0.390,10.0,-7.202,...,0.303,73.415,audio_features,0jdny0dhgjUwoIp5GkqEaA,spotify:track:0jdny0dhgjUwoIp5GkqEaA,https://api.spotify.com/v1/tracks/0jdny0dhgjUw...,https://api.spotify.com/v1/audio-analysis/0jdn...,230267.0,4.0,spotify:track:0jdny0dhgjUwoIp5GkqEaA
99996,NaN,1489,Bank Account,spotify:track:2fQrGHiQOvpL9UgPvtYy6G,Issa Album,21 Savage,0.884,0.347,8.0,-8.227,...,0.376,75.016,audio_features,2fQrGHiQOvpL9UgPvtYy6G,spotify:track:2fQrGHiQOvpL9UgPvtYy6G,https://api.spotify.com/v1/tracks/2fQrGHiQOvpL...,https://api.spotify.com/v1/audio-analysis/2fQr...,220307.0,4.0,spotify:track:2fQrGHiQOvpL9UgPvtYy6G
99997,NaN,1489,Young Dumb & Broke REMIX,spotify:track:6LxEtvMKSwmPdd5Fh1PCof,Young Dumb & Broke REMIX,Khalid,0.795,0.560,1.0,-5.773,...,0.459,136.988,audio_features,6LxEtvMKSwmPdd5Fh1PCof,spotify:track:6LxEtvMKSwmPdd5Fh1PCof,https://api.spotify.com/v1/tracks/6LxEtvMKSwmP...,https://api.spotify.com/v1/audio-analysis/6LxE...,263277.0,4.0,spotify:track:6LxEtvMKSwmPdd5Fh1PCof
99998,NaN,1489,do re mi (feat. Gucci Mane),spotify:track:53mrVsi49rLHIaKBiSvElG,do re mi (feat. Gucci Mane),blackbear,0.746,0.647,8.0,-6.388,...,0.196,112.015,audio_features,53mrVsi49rLHIaKBiSvElG,spotify:track:53mrVsi49rLHIaKBiSvElG,https://api.spotify.com/v1/tracks/53mrVsi49rLH...,https://api.spotify.com/v1/audio-analysis/53mr...,233705.0,3.0,spotify:track:53mrVsi49rLHIaKBiSvElG


In [ ]:
len(df1['pid'].unique())

1490